<a href="https://colab.research.google.com/github/aspheth/simfolder/blob/master/10_case_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datetime import datetime

def generate_report(sheet1_data, sheet2_data, sheet3_data):
    # Фильтрация студентов с рассрочкой
    clients_with_installments = []
    for client in sheet1_data:
        if client.get('installment') == 'Y':
            clients_with_installments.append([client['student_id'], client['student_name']])

    print(f"Студентов с рассрочкой: {len(clients_with_installments)}")

    # Дата отчёта
    date_rep = datetime(2023, 3, 1)

    # Должники по датам
    clients_with_debt = []
    for client in sheet2_data:
        exp_date = datetime.strptime(client['expected_payment_date'], '%d.%m.%Y')
        if exp_date >= date_rep:
            continue
        days_diff = (date_rep - exp_date).days
        n = 1 + days_diff // 183
        clients_with_debt.append([client['student_id'], n])

    print(f"Должников по датам: {len(clients_with_debt)}")

    # Расчёт долга
    result = []
    for debt in clients_with_debt:
        for client in clients_with_installments:
            for sums in sheet3_data:
                if debt[0] == client[0] == sums['student_id']:
                    missed_payments = debt[1]
                    one_payment = int(sums['one-time_payment'])
                    left_to_pay = int(sums['left_to_pay'])

                    debt_amount = missed_payments * one_payment
                    if debt_amount > left_to_pay:
                        debt_amount = left_to_pay

                    if debt_amount > 0:
                        result.append([client[1], int(debt_amount)])

    print(f"Итоговых должников: {len(result)}")

    # Сохранение файла
    with open('student_debt_report.txt', 'w', encoding='utf-8') as f:
        for name, debt in result:
            f.write(f'Студент {name} - долг {debt} рублей\n')

    print("Файл 'student_debt_report.txt' сохранён!")

# ВЫЗОВ ФУНКЦИИ (обязательно!)
generate_report(sheet1_data, sheet2_data, sheet3_data)